# Workbench client quickstart

This tutorial walks the spec §8.3 end-to-end flow with the TTI-O
Python SDK: **connect → encode → upload → query → submit a pipeline
→ download**. Every call below is exercised against a real daemon by
the `workbench-live` smoke (`python/tests/integration/test_workbench_live.py`),
so the flow is known-good rather than illustrative.

The Java SDK (`global.thalion.ttio.workbench.WorkbenchClient`) mirrors
this surface method-for-method (workplan Decision 2: Python + Java
lockstep).

## 1. Connect

`ttio.connect(url, auth=...)` authenticates and returns a
`WorkbenchClient`. Pick an auth provider for your deployment —
`PasswordTotpAuth` (interactive), `BearerAuth` (a pre-issued token),
or `BootstrapAdminAuth` (a daemon staging-root, smoke/dev only).

In [ ]:
import ttio
from ttio.workbench import PasswordTotpAuth

client = ttio.connect(
    "wss://biobank.example.org:8443/transport",
    auth=PasswordTotpAuth(username_="alice", password="…", totp="012345"),
)
print(client.session.username, sorted(client.session.capabilities))

## 2. Encode a source file to `.tio`

The `ttio encode --format <fmt>` CLI (W6.4) turns a domain file into a
`.tio` container. The same backends are reachable from Python via
`ttio.importers.registry`. Formats: `fastq | fasta | mzml | mztab |
imzml | nmrml | bam | sam | cram | thermo-raw | waters-masslynx |
bruker-timstof`.

In [ ]:
from ttio.importers import registry as import_registry

import_registry.encode("mzml", ["sample.mzML"], "sample.tio")

## 3. Upload

`upload_bytes` streams a buffered `.tio`/`.tis` to a container URI under
a project. For an encrypted upload, encrypt the channel payloads
per-AU inside a valid `.tis` (the daemon never holds a key): use
`upload_encrypted` (BYOK, caller-held key), `upload_encrypted_envelope`
(per-run DEK wrapped under a symmetric KEK), or `upload_encrypted_pqc`
(ML-KEM-1024-wrapped DEK, preview-gated). See
`docs/workbench-client/per-au-encrypted-upload-plan.md`.

In [ ]:
import asyncio

with open("sample.tio", "rb") as f:
    payload = f.read()

result = asyncio.run(client.upload_bytes(
    project="alpha",
    container_uri="uri:tio:alpha-sample",
    data=payload,
))
print(result.container_uri, result.last_acked_au_sequence)

## 4. Query

List containers in a project, or run a cohort predicate. `preview_count`
returns the match count without materialising the cohort.

In [ ]:
page = client.containers().list(project="alpha")
for c in page.containers:
    print(c.uri, c.owner, c.encrypted)

## 5. Submit a pipeline

Register (or look up) a pipeline, submit a job, and poll to a terminal
state. `jobs().events()` is an SSE long-poll alternative to polling
`jobs().get()`.

In [ ]:
import time

jobs = client.jobs()
job = jobs.submit(
    pipeline_id="eqtl-analysis",
    inputs={"container": "uri:tio:alpha-sample"},
    params={},
)
while True:
    cur = jobs.get(job.job_id)
    if cur.state in ("succeeded", "failed", "cancelled"):
        break
    time.sleep(1.0)
print("final state:", cur.state)

## 6. Download the result

`download_bytes` materialises a container (optionally with a
selective-access filter) to a buffered byte string. For an encrypted
container, the matching `download_decrypted` (BYOK),
`download_decrypted_envelope` (symmetric KEK), or
`download_decrypted_pqc` (ML-KEM) materialises the still-encrypted
`.tio` and decrypts it client-side.

In [ ]:
dl = asyncio.run(client.download_bytes(
    container_uri="uri:tio:alpha-sample-eqtl",
    filters={"chromosome": "chr6"},
))
with open("result.tio", "wb") as f:
    f.write(dl.payload)
print("wrote", len(dl.payload), "bytes")

## Federation (v1.1+)

`client.federation().peers()` lists federation peers. Against a v1.0
single-node server it returns an empty list rather than raising, so you
never have to version-detect:

In [ ]:
peers = client.federation().peers()  # [] on a v1.0 single-node server
print("federated" if client.federation().is_federated() else "single-node")